# Comprehensive Model Evaluation

Compare baseline vs fine-tuned models for both generation and embedding tasks.

## Models to Compare
1. **Generation Models**:
   - Baseline: Qwen3-4B-Instruct-2507
   - Fine-tuned: Qwen3-4B-Instruct-2507 + LoRA

2. **Embedding Models**:
   - Baseline: Qwen3-Embedding-0.6B
   - Fine-tuned: Qwen3-Embedding-0.6B + LoRA

## Metrics
- **Generation**: BLEU, ROUGE-L, BERTScore
- **Embedding**: Top-k Accuracy, MRR, NDCG@10

## 1. Setup

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score
from tqdm import tqdm
import wandb
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WANDB_PROJECT = "vietnamese-med-rag-evaluation"
WANDB_RUN = "baseline-vs-finetuned"

# Model paths
GENERATION_BASE = "Qwen/Qwen3-4B-Instruct-2507"
GENERATION_FINETUNED = "../outputs/generation_lora/finetuned_model"
EMBEDDING_BASE = "Qwen/Qwen3-Embedding-0.6B"
EMBEDDING_FINETUNED = "../outputs/embedding_lora/finetuned_model"

print(f"Using device: {DEVICE}")
print(f"\nModel paths:")
print(f"  Generation baseline: {GENERATION_BASE}")
print(f"  Generation fine-tuned: {GENERATION_FINETUNED}")
print(f"  Embedding baseline: {EMBEDDING_BASE}")
print(f"  Embedding fine-tuned: {EMBEDDING_FINETUNED}")

## 2. Initialize W&B

In [ ]:
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN,
    config={
        "generation_base": GENERATION_BASE,
        "generation_finetuned": GENERATION_FINETUNED,
        "embedding_base": EMBEDDING_BASE,
        "embedding_finetuned": EMBEDDING_FINETUNED
    }
)

print("W&B initialized successfully!")

## 3. Load Test Dataset

In [ ]:
# Load dataset
dataset = load_dataset("5CD-AI/Vietnamese-medical-LLM-QA-dataset")

# Create test split if needed
if "test" not in dataset:
    print("No test split found. Creating 80/10/10 split...")
    train_val = dataset["train"].train_test_split(test_size=0.2, seed=42)
    val_test = train_val["test"].train_test_split(test_size=0.5, seed=42)
    test_dataset = val_test["test"]
else:
    test_dataset = dataset["test"]

# Limit test set size for faster evaluation
test_dataset = test_dataset.select(range(min(500, len(test_dataset))))

print(f"Test dataset size: {len(test_dataset)}")
print(f"\nSample:")
print(f"Question: {test_dataset[0]['question']}")
print(f"Answer: {test_dataset[0]['answer'][:100]}...")

## 4. Load Generation Models

In [ ]:
print("Loading generation models...\n")

# Load baseline generation model
print("1. Loading baseline generation model...")
gen_base_tokenizer = AutoTokenizer.from_pretrained(
    GENERATION_BASE,
    trust_remote_code=True
)
if gen_base_tokenizer.pad_token is None:
    gen_base_tokenizer.pad_token = gen_base_tokenizer.eos_token

gen_base_model = AutoModelForCausalLM.from_pretrained(
    GENERATION_BASE,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)
gen_base_model.eval()
print(f"   Loaded: {GENERATION_BASE}")

# Load fine-tuned generation model
print("\n2. Loading fine-tuned generation model...")
gen_ft_tokenizer = AutoTokenizer.from_pretrained(
    GENERATION_FINETUNED,
    trust_remote_code=True
)
if gen_ft_tokenizer.pad_token is None:
    gen_ft_tokenizer.pad_token = gen_ft_tokenizer.eos_token

# Load base model then apply LoRA adapter
gen_ft_model = AutoModelForCausalLM.from_pretrained(
    GENERATION_BASE,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)
gen_ft_model = PeftModel.from_pretrained(gen_ft_model, GENERATION_FINETUNED)
gen_ft_model.eval()
print(f"   Loaded: {GENERATION_FINETUNED}")

print("\nGeneration models loaded successfully!")

## 5. Evaluate Generation Models

In [ ]:
def generate_response(model, tokenizer, question, context=None, max_new_tokens=512):
    """Generate response using chat template."""
    if context:
        user_message = f"""Dựa vào ngữ cảnh sau, hãy trả lời câu hỏi.

Ngữ cảnh: {context}

Câu hỏi: {question}"""
    else:
        user_message = f"Hãy trả lời câu hỏi sau: {question}"
    
    messages = [
        {"role": "system", "content": "Bạn là trợ lý y tế AI chuyên nghiệp."},
        {"role": "user", "content": user_message}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()


def calculate_generation_metrics(predictions, references):
    """Calculate BLEU, ROUGE-L, BERTScore."""
    # BLEU
    smoothing = SmoothingFunction().method1
    bleu_scores = []
    for pred, ref in zip(predictions, references):
        bleu = sentence_bleu([ref.split()], pred.split(), smoothing_function=smoothing)
        bleu_scores.append(bleu)
    avg_bleu = np.mean(bleu_scores)
    
    # ROUGE-L
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = []
    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)
        rouge_scores.append(score['rougeL'].fmeasure)
    avg_rouge_l = np.mean(rouge_scores)
    
    # BERTScore
    P, R, F1 = bert_score(predictions, references, lang="vi", verbose=False)
    avg_bertscore = F1.mean().item()
    
    return {
        "bleu": avg_bleu,
        "rouge_l": avg_rouge_l,
        "bertscore_f1": avg_bertscore
    }


print("Evaluating generation models...\n")

# Evaluate baseline
print("1. Baseline generation model...")
base_predictions = []
for sample in tqdm(test_dataset):
    pred = generate_response(
        gen_base_model,
        gen_base_tokenizer,
        sample["question"],
        sample.get("context", None)
    )
    base_predictions.append(pred)

references = [sample["answer"] for sample in test_dataset]
base_metrics = calculate_generation_metrics(base_predictions, references)

print("\n   Baseline Metrics:")
for metric, value in base_metrics.items():
    print(f"   - {metric}: {value:.4f}")
    wandb.log({f"generation/baseline/{metric}": value})

# Evaluate fine-tuned
print("\n2. Fine-tuned generation model...")
ft_predictions = []
for sample in tqdm(test_dataset):
    pred = generate_response(
        gen_ft_model,
        gen_ft_tokenizer,
        sample["question"],
        sample.get("context", None)
    )
    ft_predictions.append(pred)

ft_metrics = calculate_generation_metrics(ft_predictions, references)

print("\n   Fine-tuned Metrics:")
for metric, value in ft_metrics.items():
    print(f"   - {metric}: {value:.4f}")
    wandb.log({f"generation/finetuned/{metric}": value})

# Calculate improvements
print("\n3. Improvements:")
for metric in base_metrics.keys():
    improvement = ((ft_metrics[metric] - base_metrics[metric]) / base_metrics[metric]) * 100
    print(f"   - {metric}: {improvement:+.2f}%")
    wandb.log({f"generation/improvement/{metric}": improvement})

print("\nGeneration evaluation completed!")

## 6. Load Embedding Models

In [ ]:
print("Loading embedding models...\n")

# Load baseline embedding model
print("1. Loading baseline embedding model...")
emb_base_tokenizer = AutoTokenizer.from_pretrained(
    EMBEDDING_BASE,
    trust_remote_code=True,
    padding_side="left"  # CRITICAL for Qwen3-Embedding
)
if emb_base_tokenizer.pad_token is None:
    emb_base_tokenizer.pad_token = emb_base_tokenizer.eos_token

emb_base_model = AutoModel.from_pretrained(
    EMBEDDING_BASE,
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(DEVICE)
emb_base_model.eval()
print(f"   Loaded: {EMBEDDING_BASE}")

# Load fine-tuned embedding model
print("\n2. Loading fine-tuned embedding model...")
emb_ft_tokenizer = AutoTokenizer.from_pretrained(
    EMBEDDING_FINETUNED,
    trust_remote_code=True,
    padding_side="left"  # CRITICAL for Qwen3-Embedding
)
if emb_ft_tokenizer.pad_token is None:
    emb_ft_tokenizer.pad_token = emb_ft_tokenizer.eos_token

# Load base model then apply LoRA adapter
emb_ft_model = AutoModel.from_pretrained(
    EMBEDDING_BASE,
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(DEVICE)
emb_ft_model = PeftModel.from_pretrained(emb_ft_model, EMBEDDING_FINETUNED)
emb_ft_model.eval()
print(f"   Loaded: {EMBEDDING_FINETUNED}")

print("\nEmbedding models loaded successfully!")

## 7. Evaluate Embedding Models

In [ ]:
def encode_texts_for_retrieval(model, tokenizer, texts, is_query=True, batch_size=32):
    """Encode texts with Qwen3-Embedding (LEFT padding + last_token pooling)."""
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch_texts = texts[i:i+batch_size]
        
        # Format with instruction if query
        if is_query:
            task = "Tìm kiếm thông tin y tế liên quan"
            batch_texts = [f"Instruct: {task}\nQuery: {t}" for t in batch_texts]
        
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            # Last token pooling (CRITICAL)
            embeddings = outputs.last_hidden_state[:, -1, :]
        
        all_embeddings.append(embeddings.cpu().numpy())
    
    all_embeddings = np.concatenate(all_embeddings, axis=0)
    # Normalize
    all_embeddings = all_embeddings / np.linalg.norm(all_embeddings, axis=1, keepdims=True)
    
    return all_embeddings


def calculate_retrieval_metrics(similarity_matrix):
    """Calculate Top-k Accuracy, MRR, NDCG@10."""
    n_samples = similarity_matrix.shape[0]
    
    # Top-k accuracy
    metrics = {}
    for k in [1, 3, 5, 10]:
        correct = 0
        for i in range(n_samples):
            top_k_indices = np.argsort(similarity_matrix[i])[::-1][:k]
            if i in top_k_indices:
                correct += 1
        metrics[f"top_{k}_accuracy"] = correct / n_samples
    
    # MRR
    mrr_sum = 0
    for i in range(n_samples):
        sorted_indices = np.argsort(similarity_matrix[i])[::-1]
        rank = np.where(sorted_indices == i)[0][0] + 1
        mrr_sum += 1.0 / rank
    metrics["mrr"] = mrr_sum / n_samples
    
    # NDCG@10
    ndcg_scores = []
    for i in range(n_samples):
        true_relevance = np.zeros(n_samples)
        true_relevance[i] = 1
        pred_scores = similarity_matrix[i]
        ndcg = ndcg_score([true_relevance], [pred_scores], k=10)
        ndcg_scores.append(ndcg)
    metrics["ndcg@10"] = np.mean(ndcg_scores)
    
    return metrics


print("Evaluating embedding models...\n")

# Prepare data
questions = [sample["question"] for sample in test_dataset]
answers = [sample["answer"] for sample in test_dataset]

# Evaluate baseline
print("1. Baseline embedding model...")
base_q_emb = encode_texts_for_retrieval(emb_base_model, emb_base_tokenizer, questions, is_query=True)
base_a_emb = encode_texts_for_retrieval(emb_base_model, emb_base_tokenizer, answers, is_query=False)
base_sim_matrix = cosine_similarity(base_q_emb, base_a_emb)
base_emb_metrics = calculate_retrieval_metrics(base_sim_matrix)

print("\n   Baseline Metrics:")
for metric, value in base_emb_metrics.items():
    print(f"   - {metric}: {value:.4f}")
    wandb.log({f"embedding/baseline/{metric}": value})

# Evaluate fine-tuned
print("\n2. Fine-tuned embedding model...")
ft_q_emb = encode_texts_for_retrieval(emb_ft_model, emb_ft_tokenizer, questions, is_query=True)
ft_a_emb = encode_texts_for_retrieval(emb_ft_model, emb_ft_tokenizer, answers, is_query=False)
ft_sim_matrix = cosine_similarity(ft_q_emb, ft_a_emb)
ft_emb_metrics = calculate_retrieval_metrics(ft_sim_matrix)

print("\n   Fine-tuned Metrics:")
for metric, value in ft_emb_metrics.items():
    print(f"   - {metric}: {value:.4f}")
    wandb.log({f"embedding/finetuned/{metric}": value})

# Calculate improvements
print("\n3. Improvements:")
for metric in base_emb_metrics.keys():
    improvement = ((ft_emb_metrics[metric] - base_emb_metrics[metric]) / base_emb_metrics[metric]) * 100
    print(f"   - {metric}: {improvement:+.2f}%")
    wandb.log({f"embedding/improvement/{metric}": improvement})

print("\nEmbedding evaluation completed!")

## 8. Visualize Results

In [ ]:
# Create comparison dataframes
gen_comparison = pd.DataFrame({
    "Metric": list(base_metrics.keys()),
    "Baseline": list(base_metrics.values()),
    "Fine-tuned": list(ft_metrics.values())
})

emb_comparison = pd.DataFrame({
    "Metric": list(base_emb_metrics.keys()),
    "Baseline": list(base_emb_metrics.values()),
    "Fine-tuned": list(ft_emb_metrics.values())
})

# Plot generation metrics
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Generation comparison
ax1 = axes[0]
x = np.arange(len(gen_comparison))
width = 0.35
ax1.bar(x - width/2, gen_comparison["Baseline"], width, label="Baseline", alpha=0.8)
ax1.bar(x + width/2, gen_comparison["Fine-tuned"], width, label="Fine-tuned", alpha=0.8)
ax1.set_xlabel("Metric")
ax1.set_ylabel("Score")
ax1.set_title("Generation Model Comparison")
ax1.set_xticks(x)
ax1.set_xticklabels(gen_comparison["Metric"], rotation=45)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# Embedding comparison
ax2 = axes[1]
x = np.arange(len(emb_comparison))
ax2.bar(x - width/2, emb_comparison["Baseline"], width, label="Baseline", alpha=0.8)
ax2.bar(x + width/2, emb_comparison["Fine-tuned"], width, label="Fine-tuned", alpha=0.8)
ax2.set_xlabel("Metric")
ax2.set_ylabel("Score")
ax2.set_title("Embedding Model Comparison")
ax2.set_xticks(x)
ax2.set_xticklabels(emb_comparison["Metric"], rotation=45)
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/model_comparison.png", dpi=300, bbox_inches="tight")
wandb.log({"comparison_plot": wandb.Image("../outputs/model_comparison.png")})
plt.show()

print("Visualization saved to ../outputs/model_comparison.png")

## 9. Summary Report

In [ ]:
print("\n" + "="*80)
print("COMPREHENSIVE EVALUATION SUMMARY")
print("="*80)

print("\n[GENERATION MODEL]")
print("-" * 80)
print(f"{'Metric':<20} {'Baseline':>15} {'Fine-tuned':>15} {'Improvement':>15}")
print("-" * 80)
for metric in base_metrics.keys():
    base_val = base_metrics[metric]
    ft_val = ft_metrics[metric]
    improvement = ((ft_val - base_val) / base_val) * 100
    print(f"{metric:<20} {base_val:>15.4f} {ft_val:>15.4f} {improvement:>14.2f}%")

print("\n[EMBEDDING MODEL]")
print("-" * 80)
print(f"{'Metric':<20} {'Baseline':>15} {'Fine-tuned':>15} {'Improvement':>15}")
print("-" * 80)
for metric in base_emb_metrics.keys():
    base_val = base_emb_metrics[metric]
    ft_val = ft_emb_metrics[metric]
    improvement = ((ft_val - base_val) / base_val) * 100
    print(f"{metric:<20} {base_val:>15.4f} {ft_val:>15.4f} {improvement:>14.2f}%")

print("\n" + "="*80)

# Save summary to file
with open("../outputs/evaluation_summary.txt", "w") as f:
    f.write("COMPREHENSIVE EVALUATION SUMMARY\n")
    f.write("=" * 80 + "\n\n")
    
    f.write("[GENERATION MODEL]\n")
    f.write("-" * 80 + "\n")
    f.write(f"{'Metric':<20} {'Baseline':>15} {'Fine-tuned':>15} {'Improvement':>15}\n")
    f.write("-" * 80 + "\n")
    for metric in base_metrics.keys():
        base_val = base_metrics[metric]
        ft_val = ft_metrics[metric]
        improvement = ((ft_val - base_val) / base_val) * 100
        f.write(f"{metric:<20} {base_val:>15.4f} {ft_val:>15.4f} {improvement:>14.2f}%\n")
    
    f.write("\n[EMBEDDING MODEL]\n")
    f.write("-" * 80 + "\n")
    f.write(f"{'Metric':<20} {'Baseline':>15} {'Fine-tuned':>15} {'Improvement':>15}\n")
    f.write("-" * 80 + "\n")
    for metric in base_emb_metrics.keys():
        base_val = base_emb_metrics[metric]
        ft_val = ft_emb_metrics[metric]
        improvement = ((ft_val - base_val) / base_val) * 100
        f.write(f"{metric:<20} {base_val:>15.4f} {ft_val:>15.4f} {improvement:>14.2f}%\n")

print("Summary saved to ../outputs/evaluation_summary.txt")

# Finish W&B
wandb.finish()

print("\nComprehensive evaluation completed!")